[Reference](https://medium.com/activated-thinker/predicting-movie-success-using-machine-learning-b804bb5e7908)

# Loading the Dataset in Colab

In [1]:
!pip install kagglehub

import kagglehub

# Download latest version
path = kagglehub.dataset_download("therealsampat/predict-movie-success-rate")

print("Path to dataset files:", path)

import pandas as pd
import os

os.listdir(path)

100%|██████████| 119k/119k [00:00<00:00, 49.1MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/therealsampat/predict-movie-success-rate/versions/1


['movie_success_rate.csv']

In [2]:
df = pd.read_csv(os.path.join(path, "movie_success_rate.csv"))
df.head()

,Rank,Title,Genre,Description,Director,Actors,Year,Runtime (Minutes),Rating,Votes,...,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western,Success
0,1.0,Guardians of the Galaxy,"Action,Adventure,Sci-Fi",A group of intergalactic criminals are forced ...,James Gunn,"Chris Pratt, Vin Diesel, Bradley Cooper, Zoe S...",2014.0,121.0,8.1,757074.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,2.0,Prometheus,"Adventure,Mystery,Sci-Fi","Following clues to the origin of mankind, a te...",Ridley Scott,"Noomi Rapace, Logan Marshall-Green, Michael Fa...",2012.0,124.0,7.0,485820.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,3.0,Split,"Horror,Thriller",Three girls are kidnapped by a man with a diag...,M. Night Shyamalan,"James McAvoy, Anya Taylor-Joy, Haley Lu Richar...",2016.0,117.0,7.3,157606.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4.0,Sing,"Animation,Comedy,Family","In a city of humanoid animals, a hustling thea...",Christophe Lourdelet,"Matthew McConaughey,Reese Witherspoon, Seth Ma...",2016.0,108.0,7.2,60545.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,Suicide Squad,"Action,Adventure,Fantasy",A secret government agency recruits some of th...,David Ayer,"Will Smith, Jared Leto, Margot Robbie, Viola D...",2016.0,123.0,6.2,393727.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Rank                838 non-null    float64
 1   Title               838 non-null    object 
 2   Genre               838 non-null    object 
 3   Description         838 non-null    object 
 4   Director            838 non-null    object 
 5   Actors              838 non-null    object 
 6   Year                838 non-null    float64
 7   Runtime (Minutes)   838 non-null    float64
 8   Rating              839 non-null    float64
 9   Votes               839 non-null    float64
 10  Revenue (Millions)  839 non-null    float64
 11  Metascore           838 non-null    float64
 12  Action              838 non-null    float64
 13  Adventure           838 non-null    float64
 14  Aniimation          838 non-null    float64
 15  Biography           838 non-null    float64
 16  Comedy  

In [4]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = df.dropna().reset_index(drop=True)

# Fix column names for convenience
df.columns = df.columns.str.strip()

# Why Actor and Director Names Matter

In [5]:
def split_people(x):
    return [i.strip() for i in x.split(",")]

df["Actors_List"] = df["Actors"].apply(split_people)
df["Director_List"] = df["Director"].apply(split_people)

actor_success = {}

for actors, success in zip(df["Actors_List"], df["Success"]):
    for actor in actors:
        actor_success.setdefault(actor, []).append(success)

actor_success_score = {
    actor: np.mean(scores) for actor, scores in actor_success.items()
}

director_success = {}

for directors, success in zip(df["Director_List"], df["Success"]):
    for director in directors:
        director_success.setdefault(director, []).append(success)

director_success_score = {
    director: np.mean(scores) for director, scores in director_success.items()
}

In [6]:
def aggregate_actor_score(actors):
    scores = [actor_success_score.get(a, 0.5) for a in actors]
    return np.mean(scores)

def aggregate_director_score(directors):
    scores = [director_success_score.get(d, 0.5) for d in directors]
    return np.mean(scores)

df["Actor_Score"] = df["Actors_List"].apply(aggregate_actor_score)
df["Director_Score"] = df["Director_List"].apply(aggregate_director_score)

# Automatically Selecting Genre Features

In [7]:
genre_cols = [
    col for col in df.columns
    if col not in [
        "Rank", "Title", "Genre", "Description", "Director", "Actors",
        "Actors_List", "Director_List", "Success"
    ]
    and df[col].nunique() <= 2
    and df[col].dtype != object
]

# Final Feature Set and Target

In [8]:
features = (
    ["Year", "Runtime (Minutes)", "Actor_Score", "Director_Score"]
    + genre_cols
)

X = df[features]
y = df["Success"]

# Training the Model

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=300,
                       random_state=42)

# Evaluation and What the Metrics Mean

In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9702380952380952
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.98       138
         1.0       0.90      0.93      0.92        30

    accuracy                           0.97       168
   macro avg       0.94      0.96      0.95       168
weighted avg       0.97      0.97      0.97       168



# Saving Artifacts for Deployment

In [12]:
import pickle
import os

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/movie_success_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("artifacts/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("artifacts/actor_success.pkl", "wb") as f:
    pickle.dump(actor_success_score, f)
with open("artifacts/director_success.pkl", "wb") as f:
    pickle.dump(director_success_score, f)
with open("artifacts/features.pkl", "wb") as f:
    pickle.dump(features, f)